# Edit distance search
Manipulating inference results, creating training and testing labels

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd

In [ ]:
with open(r'C:\Users\Parv\Doc\HelixWorks\Basecalling\code\motifcaller\data\empirical\edit_distance_motif_search\res-loose-chain.txt', 'r') as f:
    lines = f.readlines()

def clean_prediction(edit_motif_search_prediction):

    cleaned_prediction = []
    for i in edit_motif_search_prediction:
        if i == 'fake':
            cleaned_prediction.append([])
        else:
            cleaned_prediction.append([int(i[1])])

    return cleaned_prediction
        

motif_predictions = []
orientations = []
read_ids = []

for line in lines:
    #print(line.split())
    split_line = line.split()
    read_id = split_line[0][3:]
    orientation = split_line[1]
    prediction = split_line[4][8:].split('->')
    if not (prediction[0].startswith('f') or prediction[0].startswith('m')):
        prediction = prediction[1:]

    cleaned_prediction = clean_prediction(prediction)
    
    motif_predictions.append(cleaned_prediction)
    orientations.append(orientation)
    read_ids.append(read_id)


df = pd.DataFrame({"read_id": read_ids, "orientation": orientations, "motif_seq": motif_predictions})


In [ ]:
df.to_pickle(r"C:\Users\Parv\Doc\HelixWorks\Basecalling\code\motifcaller\data\empirical\edit_distance_motif_search\edit_distance_motif_search.pkl")

## Balancing edit-train df

In [51]:
from data_functions import get_cleaned_encoded_file

In [50]:
encoded_df = pd.read_csv(r"C:\Users\Parv\Doc\HelixWorks\Basecalling\code\motifcaller\data\empirical\EIC01-01-1280-T1_encoded.tsv", sep='\t')

In [52]:
t = get_cleaned_encoded_file(encoded_df)

In [54]:
t = t[['ONT_Barcode', 'HW_Address', 'payload']]

In [4]:
edit_train_df = pd.read_pickle(r"C:\Users\Parv\Doc\HelixWorks\Basecalling\code\motifcaller\data\empirical\full_datasets\edit_master_train.pkl")

In [58]:
edit_train_df = edit_train_df.drop(columns=['payload'])

In [61]:
edit_train_df.columns

Index(['read_id', 'ONT_Barcode', 'HW_Address', 'orientation', 'start_end',
       'library_motif', 'squiggle', 'motif_seq', 'strand',
       'payload_motifs_found', 'edit_spacer_seq', 'edit_motifs_found'],
      dtype='object')

In [62]:
t

,ONT_Barcode,HW_Address,payload
0,1,barcode_external01_internal01,"[[2, 3, 4, 5], [1, 2, 7, 8], [1, 4, 5, 6], [4,..."
1,1,barcode_external02_internal01,"[[3, 4, 7, 8], [2, 3, 4, 8], [2, 4, 6, 7], [1,..."
2,1,barcode_external03_internal01,"[[1, 3, 5, 8], [2, 4, 5, 6], [1, 5, 6, 8], [1,..."
3,1,barcode_external04_internal01,"[[1, 4, 5, 8], [3, 4, 7, 8], [1, 4, 7, 8], [1,..."
4,1,barcode_external05_internal01,"[[2, 4, 5, 6], [3, 4, 6, 7], [4, 5, 6, 8], [1,..."
...,...,...,...
1275,77,barcode_external04_internal08,"[[1, 2, 3, 6], [2, 4, 5, 7], [1, 3, 4, 7], [1,..."
1276,77,barcode_external05_internal08,"[[4, 6, 7, 8], [1, 2, 4, 8], [3, 4, 5, 7], [2,..."
1277,77,barcode_external06_internal08,"[[1, 4, 7, 8], [4, 6, 7, 8], [1, 2, 4, 8], [2,..."
1278,77,barcode_external07_internal08,"[[1, 2, 3, 4], [1, 2, 3, 4], [1, 2, 3, 4], [1,..."


In [65]:
edit_train_df['ONT_Barcode'] = edit_train_df['ONT_Barcode'].apply(lambda x: int(x[-2:]))

In [66]:
merged_df = pd.merge(edit_train_df, t, on=['ONT_Barcode', 'HW_Address'])

Steps
1. Filter out reads where more than 8 motifs are identified
2. Keep the ones with no error
3. For the ones with error, check motif search label and encoded to make the perfect label with no errors
4. Double check that there are no errors for the whole dataset

In [146]:
filtered_df = merged_df.loc[merged_df['payload_motifs_found'] > 7]

In [129]:
from transcript_sorting import sort_transcript
from utils import evaluate_prediction, create_spacer_sequence

In [147]:
filtered_df['edit_payload_seq'] = filtered_df['edit_spacer_seq'].apply(lambda x: sort_transcript(x))

C:\Users\Parv\AppData\Local\Temp\ipykernel_4900\3848269342.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['edit_payload_seq'] = filtered_df['edit_spacer_seq'].apply(lambda x: sort_transcript(x))


In [148]:
def remove_erroneous_motifs(prediction, original):
    corrected = [[] for i in range(8)]

    cycle_num = 0
    for i, j in zip(prediction, original):
        for k in i:
            if k in j:
                corrected[cycle_num].append(k)
        cycle_num += 1
    return corrected

In [163]:
counter = 0
mf = 0
me = 0
corrected_edit_payload_seq = []

for ind, row in filtered_reverse.iterrows():
    i = row['edit_payload_seq_no_error']
    j = row['payload']
    corrected = remove_erroneous_motifs(i, j)
    metrics = evaluate_prediction(corrected, j)
    mf += metrics[0]
    me += metrics[1]
    corrected_edit_payload_seq.append(corrected)
    


In [152]:
filtered_df['edit_payload_seq_no_error'] = corrected_edit_payload_seq

C:\Users\Parv\AppData\Local\Temp\ipykernel_4900\819696354.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['edit_payload_seq_no_error'] = corrected_edit_payload_seq


In [153]:
filtered_df['edit_spacer_seq'] = filtered_df['edit_payload_seq_no_error'].apply(lambda x: create_spacer_sequence(x))

C:\Users\Parv\AppData\Local\Temp\ipykernel_4900\1405600288.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['edit_spacer_seq'] = filtered_df['edit_payload_seq_no_error'].apply(lambda x: create_spacer_sequence(x))


In [156]:
filtered_df

,read_id,ONT_Barcode,HW_Address,orientation,start_end,library_motif,squiggle,motif_seq,strand,payload_motifs_found,edit_spacer_seq,edit_motifs_found,payload,edit_payload_seq,edit_payload_seq_no_error
660,1361a5db-d135-4e98-bb49-7a53c8d72991,1,barcode_external01_internal01,-|-|-|-|-|-|-|-,541-590|491-540|392-441|342-391|292-341|242-29...,ltm8_1x1|ltm8_2x1|ltm8_4x2|ltm8_5x7|ltm8_6x4|l...,"[671, 498, 504, 526, 502, 512, 527, 528, 517, ...","[17, 5, 17, 16, 6, 16, 15, 5, 15, 14, 4, 14, 1...",-,8,"[12, 2, 12, 14, 4, 14, 15, 5, 15, 16, 6, 16, 1...",10.0,"[[2, 3, 4, 5], [1, 2, 7, 8], [1, 4, 5, 6], [4,...","[[1], [2], [7], [4], [5], [6], [5], [1]]","[[], [2], [], [4], [5], [6], [5], [1]]"
661,1361a5db-d135-4e98-bb49-7a53c8d72991,1,barcode_external01_internal01,-|-|-|-|-|-|-|-,541-590|491-540|392-441|342-391|292-341|242-29...,ltm8_1x1|ltm8_2x1|ltm8_4x2|ltm8_5x7|ltm8_6x4|l...,"[671, 498, 504, 526, 502, 512, 527, 528, 517, ...","[17, 5, 17, 16, 6, 16, 15, 5, 15, 14, 4, 14, 1...",-,8,"[12, 2, 12, 14, 4, 14, 15, 5, 15, 16, 6, 16, 1...",10.0,"[[2, 3, 4, 5], [1, 2, 7, 8], [1, 4, 5, 6], [4,...","[[1], [2], [7], [4], [5], [6], [5], [1]]","[[], [2], [], [4], [5], [6], [5], [1]]"
662,1361a5db-d135-4e98-bb49-7a53c8d72991,1,barcode_external01_internal01,-|-|-|-|-|-|-|-,541-590|491-540|392-441|342-391|292-341|242-29...,ltm8_1x1|ltm8_2x1|ltm8_4x2|ltm8_5x7|ltm8_6x4|l...,"[671, 498, 504, 526, 502, 512, 527, 528, 517, ...","[17, 5, 17, 16, 6, 16, 15, 5, 15, 14, 4, 14, 1...",-,8,"[12, 2, 12, 14, 4, 14, 15, 5, 15, 16, 6, 16, 1...",10.0,"[[2, 3, 4, 5], [1, 2, 7, 8], [1, 4, 5, 6], [4,...","[[1], [2], [7], [4], [5], [6], [5], [1]]","[[], [2], [], [4], [5], [6], [5], [1]]"
663,1361a5db-d135-4e98-bb49-7a53c8d72991,1,barcode_external01_internal01,-|-|-|-|-|-|-|-,541-590|491-540|392-441|342-391|292-341|242-29...,ltm8_1x1|ltm8_2x1|ltm8_4x2|ltm8_5x7|ltm8_6x4|l...,"[671, 498, 504, 526, 502, 512, 527, 528, 517, ...","[17, 5, 17, 16, 6, 16, 15, 5, 15, 14, 4, 14, 1...",-,8,"[12, 2, 12, 14, 4, 14, 15, 5, 15, 16, 6, 16, 1...",10.0,"[[2, 3, 4, 5], [1, 2, 7, 8], [1, 4, 5, 6], [4,...","[[1], [2], [7], [4], [5], [6], [5], [1]]","[[], [2], [], [4], [5], [6], [5], [1]]"
664,1361a5db-d135-4e98-bb49-7a53c8d72991,1,barcode_external01_internal01,-|-|-|-|-|-|-|-,541-590|491-540|392-441|342-391|292-341|242-29...,ltm8_1x1|ltm8_2x1|ltm8_4x2|ltm8_5x7|ltm8_6x4|l...,"[671, 498, 504, 526, 502, 512, 527, 528, 517, ...","[17, 5, 17, 16, 6, 16, 15, 5, 15, 14, 4, 14, 1...",-,8,"[12, 2, 12, 14, 4, 14, 15, 5, 15, 16, 6, 16, 1...",10.0,"[[2, 3, 4, 5], [1, 2, 7, 8], [1, 4, 5, 6], [4,...","[[1], [2], [7], [4], [5], [6], [5], [1]]","[[], [2], [], [4], [5], [6], [5], [1]]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9243795,fceeaedc-11b6-4ce3-b678-4685ab3b8c80,77,barcode_external08_internal08,+|+|+|+|+|+|+|+,220-269|270-319|320-369|370-419|420-469|470-51...,ltm8_3x2|ltm8_4x4|ltm8_5x3|ltm8_6x2|ltm8_7x4|l...,"[561, 507, 537, 577, 495, 452, 488, 488, 459, ...","[11, 2, 11, 12, 4, 12, 13, 3, 13, 14, 2, 14, 1...",+,8,[],1.0,"[[1, 2, 3, 4], [1, 2, 3, 4], [1, 2, 3, 4], [1,...","[[], [], [], [], [], [], [7], []]","[[], [], [], [], [], [], [], []]"
9243796,fceeaedc-11b6-4ce3-b678-4685ab3b8c80,77,barcode_external08_internal08,+|+|+|+|+|+|+|+,220-269|270-319|320-369|370-419|420-469|470-51...,ltm8_3x2|ltm8_4x4|ltm8_5x3|ltm8_6x2|ltm8_7x4|l...,"[561, 507, 537, 577, 495, 452, 488, 488, 459, ...","[11, 2, 11, 12, 4, 12, 13, 3, 13, 14, 2, 14, 1...",+,8,[],1.0,"[[1, 2, 3, 4], [1, 2, 3, 4], [1, 2, 3, 4], [1,...","[[], [], [], [], [], [], [7], []]","[[], [], [], [], [], [], [], []]"
9243797,fceeaedc-11b6-4ce3-b678-4685ab3b8c80,77,barcode_external08_internal08,+|+|+|+|+|+|+|+,220-269|270-319|320-369|370-419|420-469|470-51...,ltm8_3x2|ltm8_4x4|ltm8_5x3|ltm8_6x2|ltm8_7x4|l...,"[561, 507, 537, 577, 495, 452, 488, 488, 459, ...","[11, 2, 11, 12, 4, 12, 13, 3, 13, 14, 2, 14, 1...",+,8,[],1.0,"[[1, 2, 3, 4], [1, 2, 3, 4], [1, 2, 3, 4], [1,...","[[], [], [], [], [], [], [7], []]","[[], [], [], [], [], [],

In [157]:
filtered_reverse = filtered_df.loc[filtered_df['orientation'].str.startswith('-')]

In [160]:
filtered_reverse['edit_spacer_seq'] = filtered_reverse['edit_spacer_seq'].apply(lambda x: x[::-1])

C:\Users\Parv\AppData\Local\Temp\ipykernel_4900\2773630298.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_reverse['edit_spacer_seq'] = filtered_reverse['edit_spacer_seq'].apply(lambda x: x[::-1])


In [ ]:
filtered_forward['edit_spacer_seq'] = filtered_df['edit_payload_seq_no_error'].apply()

In [166]:
filtered_reverse.to_pickle(r"C:\Users\Parv\Doc\HelixWorks\Basecalling\code\motifcaller\data\empirical\edit_distance_motif_search\edit_train_filtered_reverse.pkl")

In [167]:
filtered_forward = pd.read_pickle(r"C:\Users\Parv\Doc\HelixWorks\Basecalling\code\motifcaller\data\empirical\edit_distance_motif_search\edit_train_filtered_forward.pkl")